# 🎬 YT Short Clipper Pro — Colab

Run everything in 2 cells. Just click ▶️ and wait for the Ngrok URL.

**Features**: AI Analysis, Face Tracking, Karaoke Subtitle, B-Roll Overlay, Background Music

## 1. Install YT-Short-Clipper-Offline

In [ ]:
#@title 1. Install YT-Short-Clipper-Offline { display-mode: "form" }
#@markdown Clone repo and install all dependencies.

import subprocess, sys, os

print("="*60)
print("📦 Installing YT-Short-Clipper-Offline...")
print("="*60)

# --- Clone or pull repo ---
repo_url = "https://github.com/Chukie99/yt-short-clipper-offline.git"
repo_dir = "/content/yt-short-clipper-offline"

if os.path.exists(repo_dir):
    print("\n[1/4] Repository exists, pulling latest...")
    subprocess.run(["git", "-C", repo_dir, "pull"], capture_output=True)
else:
    print("\n[1/4] Cloning repository...")
    subprocess.run(["git", "clone", repo_url, repo_dir], capture_output=True)

os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
print("  ✅ Repository ready!")

# --- Install system dependencies ---
print("\n[2/4] Installing system dependencies...")
subprocess.run(["apt-get", "-qq", "install", "ffmpeg"], capture_output=True)
print("  ✅ FFmpeg installed!")

# --- Install Python dependencies ---
print("\n[3/4] Installing Python dependencies...")
deps = [
    "yt-dlp[default]", "opencv-python-headless", "numpy", "Pillow",
    "requests", "mediapipe", "python-dotenv", "faster-whisper",
    "google-genai", "streamlit", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, capture_output=True)
print("  ✅ Python packages installed!")

# --- Verify installation ---
print("\n[4/4] Verifying installation...")
for f in ["clipper_core.py", "app.py", "requirements.txt"]:
    status = "✅" if os.path.exists(os.path.join(repo_dir, f)) else "❌"
    print(f"  {status} {f}")

print("\n" + "="*60)
print("YT-Short-Clipper is ready!")
print("="*60)

## 2. Configure Colab Tunnel & Launch WebUI

In [ ]:
#@title 2. Configure Colab Tunnel & Launch WebUI { display-mode: "form" }
#@markdown Configure API keys and launch Streamlit with Ngrok tunnel.

import os, sys, time, subprocess
from pathlib import Path

# --- Colab Form Parameters ---
ngrok_authtoken = "" #@param {type:"string"}
ai_provider = "Gemini" #@param ["Gemini", "Groq", "OpenRouter"]
api_key = "" #@param {type:"string"}

# --- Change to repo directory ---
repo_dir = "/content/yt-short-clipper-offline"
os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# --- Setup directories ---
from clipper_core import setup_directories
setup_directories(
    temp_dir="/content/temp",
    output_dir="/content/drive/MyDrive/YTShortClipper/output",
    config_file="/content/drive/MyDrive/YTShortClipper/config.json",
)

# --- Export config to .env ---
config = {
    "ai_provider": ai_provider,
    "gemini_api_key": api_key if ai_provider == "Gemini" else "",
    "groq_api_key": api_key if ai_provider == "Groq" else "",
    "openrouter_api_key": api_key if ai_provider == "OpenRouter" else "",
    "ngrok_authtoken": ngrok_authtoken,
}

env_path = Path(repo_dir) / ".env"
with open(env_path, "w") as f:
    for key, val in config.items():
        f.write(f"{key.upper()}={val}\n")

print("✅ Config exported to .env")

# --- Initialize Ngrok ---
public_url = None
if ngrok_authtoken:
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = ngrok_authtoken
        ngrok.kill()
        tunnel = ngrok.connect(8501, "http")
        public_url = tunnel.public_url
        print(f"✅ Ngrok tunnel created: {public_url}")
    except Exception as e:
        print(f"⚠️ Ngrok error: {e}")
else:
    print("⚠️ No Ngrok token provided. Using Colab direct link.")

# --- Launch Streamlit ---
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.address=0.0.0.0",
     "--browser.gatherUsageStats=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Waiting for Streamlit to start...")
for i in range(20):
    time.sleep(1)
    if process.poll() is not None:
        print(f"❌ Streamlit crashed! Exit code: {process.returncode}")
        break
else:
    print("✅ Streamlit is running!")

# --- Show public URL ---
print("\n" + "="*60)
if public_url:
    print(f"YT-Short-Clipper-Offline is ready:")
    print(public_url)
else:
    print("Colab Direct Link:")
    print("http://localhost:8501")
print("="*60)
print("\nOpen the URL above in your browser to access the WebUI!")
print("To stop: run 'pkill -f streamlit' in a new cell\n")